In [0]:
%pip install databricks-vectorsearch databricks-sdk langchain mlflow mlflow[databricks] langchain-community


In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow.pyfunc

class LCELRAGModel(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        import os
        from langchain_core.runnables import RunnablePassthrough
        from langchain_core.prompts import PromptTemplate
        from langchain_community.chat_models import ChatDatabricks
        from databricks.vector_search.client import VectorSearchClient
        from langchain_community.vectorstores import DatabricksVectorSearch
        from langchain_community.embeddings import DatabricksEmbeddings
        
        workspace_url = os.environ.get("DATABRICKS_HOST")
        token = os.environ.get("DATABRICKS_TOKEN")
        vector_search_endpoint = os.environ.get("VECTOR_SEARCH_ENDPOINT")
        vector_search_index = os.environ.get("VECTOR_SEARCH_INDEX")

        def get_retriever():

            embedding_model = DatabricksEmbeddings(endpoint= 'databricks-bge-large-en')

            client = VectorSearchClient(
                workspace_url=workspace_url,
                personal_access_token=token
            )
            vs_index = client.get_index(endpoint_name=vector_search_endpoint, index_name = vector_search_index)

            vector_store = DatabricksVectorSearch(
                vs_index,
                embedding=embedding_model,
                # The column name in the index that contains the text data to be embedded
                text_column="text"
            )
            return vector_store.as_retriever()


        llm = ChatDatabricks(
            endpoint="databricks-meta-llama-3-3-70b-instruct"
        )

        TEMPLATE = """You are an assistant for home appliance users. You are answering how to, maintenance and troubleshooting questions regarding the appliances you have data on. If the question is not related to one of these topics, kindly decline to answer. If you don't know the answer, just say that you don't know, don't try to make up an answer. If the question appears to be for an appliance you don't have data on, say so. Keep the answer as concise as possible. Provide all answers only in English. Use the following pieces of context to answer the question at the end: 
        {context} 
        Question: {question} 
        Answer:
                   """

        prompt = PromptTemplate.from_template(TEMPLATE)

        retriever = get_retriever()

        self.chain = (
            {
                "context": retriever,
                "question": RunnablePassthrough(),
            }
            | prompt
            | llm
        )

    def predict(self, context, model_input):
        query = model_input["query"].iloc[0]

        result = self.chain.invoke(query)

        return [result.content]

In [0]:
import mlflow
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

# Example input/output
input_example = {"query": "What is control panel?"}
output_example = "Control panel is used to operate the appliance."

signature = infer_signature(input_example, output_example)

with mlflow.start_run():
    mlflow.pyfunc.log_model(
        artifact_path="rag_model",
        python_model=LCELRAGModel(),
        registered_model_name="llm.rag.appliance_chatbot_model",
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            "mlflow",
            "langchain",
            "langchain-core",
            "langchain-community",
            "databricks-vectorsearch",
            "typing_extensions>=4.9.0"        ]
    )

In [0]:

import os
import mlflow
os.environ["DATABRICKS_HOST"] = dbutils.secrets.get('rag_llm_scope', 'db_host')
os.environ["DATABRICKS_TOKEN"] = dbutils.secrets.get('rag_llm_scope', 'db_agent_app_token')
os.environ["VECTOR_SEARCH_ENDPOINT"] =  'agent_db'
os.environ["VECTOR_SEARCH_INDEX"] = 'llm.rag.idx_docs_text'
model = mlflow.pyfunc.load_model("models:/llm.rag.appliance_chatbot_model/13")

model.predict({"query": "What is Control Panel ?"})